# BengaluruFlow — Exploratory Data Analysis

This notebook explores the Bangalore Traffic Pulse dataset before modelling.
The ML work (LSTM, Transformer, Autoencoder) lives in `src/` — this is just EDA.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_data, check_no_nulls

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

In [ ]:
df = load_data()
check_no_nulls(df)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic statistics
df.describe().round(2)

In [ ]:
# Column data types
df.dtypes

In [ ]:
# Distribution of numeric columns
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols].hist(bins=30, figsize=(16, 10), color='steelblue', edgecolor='white')
plt.suptitle('Numeric Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Traffic Volume over time (all roads combined)
daily_volume = df.groupby('Date')['Traffic Volume'].mean()

plt.figure(figsize=(14, 4))
daily_volume.plot(color='#1f77b4')
plt.title('Mean Daily Traffic Volume (all roads)')
plt.xlabel('Date')
plt.ylabel('Traffic Volume')
plt.tight_layout()
plt.show()

In [ ]:
# Traffic volume by Area
area_vol = df.groupby('Area Name')['Traffic Volume'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
area_vol.plot(kind='bar', color=sns.color_palette('tab10', len(area_vol)))
plt.title('Average Traffic Volume by Area')
plt.xlabel('Area')
plt.ylabel('Avg Traffic Volume')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            annot_kws={'size': 8}, linewidths=0.5)
plt.title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Day of week vs traffic volume
df['day_of_week'] = df['Date'].dt.dayofweek
dow_map = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}
df['day_name'] = df['day_of_week'].map(dow_map)

plt.figure(figsize=(8, 4))
dow_vol = df.groupby('day_of_week')['Traffic Volume'].mean()
plt.bar([dow_map[i] for i in range(7)], [dow_vol[i] for i in range(7)],
        color=sns.color_palette('husl', 7))
plt.title('Average Traffic Volume by Day of Week')
plt.xlabel('Day')
plt.ylabel('Avg Traffic Volume')
plt.tight_layout()
plt.show()

In [ ]:
# Weather vs Congestion
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x='Weather Conditions', y='Congestion Level',
            palette='Set2')
plt.title('Congestion Level by Weather Condition')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Incident Reports distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['Incident Reports'], bins=20, kde=False, color='#d62728')
plt.title('Incident Reports Distribution')
plt.xlabel('Incident Reports')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

pct_with_incident = 100 * (df['Incident Reports'] > 0).mean()
print(f'Rows with Incident Reports > 0: {pct_with_incident:.1f}%')
print('(These are used as weak anomaly labels in the autoencoder evaluation)')